# L11 · 向量范数 — L1 / L2 / ∞ 范数的计算、几何形状与正则化含义

**目标**：会算向量长度(L2 范数)、向量间距离，会把向量归一化成单位长度。

**为什么对 Aurora 重要**：MSE 损失就是误差向量的 L2 范数平方；L1/L2 正则项直接约束权重向量的范数；MFCC 特征送入模型前先按 L2 范数归一化，让不同帧的能量落在同一尺度。

## 本课剧情：给箭头量身高

范数把「向量有多长」变成一个数：L2 范数是各分量平方和再开根，几何上等于欧氏距离；L1 范数是各分量绝对值之和，走的是棋盘格路程。归一化把向量除以自身 L2 范数，方向不变，长度压到 1，不同尺度的特征因此可以直接比较。

## 1. L2 范数 = 向量的长度（勾股定理推广）

`‖v‖₂ = sqrt(v₁² + v₂² + ⋯ + vₙ²)`

L2 范数归一化把向量投影到单位球面上，使得之后的点积只测量方向相似度，不受幅度影响。在说话人识别里，每段音频编码成 d-vector 后先做 L2 归一化，比对时只比较方向——两个人说话风格相似、音量不同，归一化后的 d-vector 夹角仍然很小。

## 符号入口：先看范数，再看归一化

L2 范数写作 `‖v‖₂`，L1 写作 `‖v‖₁`。给向量 `v = [3, 4]`：`‖v‖₂ = sqrt(9+16) = 5`，`‖v‖₁ = 3+4 = 7`。两个范数都量「长度」，但 L2 对大分量惩罚更重，L1 对各分量一视同仁。

In [ ]:
import numpy as np
v = np.array([3.0, 4.0])
print('手算:', np.sqrt(np.sum(v**2)))   # 5.0
print('numpy:', np.linalg.norm(v))      # 5.0
print('L1 范数(绝对值之和):', np.sum(np.abs(v)))

## 动手观察：同一向量，两种长度

运行下方代码，注意 L1 和 L2 给出的数值不同，但归一化后向量的 L2 范数恰好等于 1。改动某个分量的大小，观察哪个范数变化更剧烈。

In [ ]:
import numpy as np

v = np.array([3.0, 4.0])
A = np.array([[2.0, 0.0],
              [0.0, 0.5]])

print('v =', v, 'shape =', v.shape)
print('A =')
print(A)
print('A shape =', A.shape)
print('A @ v =', A @ v)
print('向量长度 ||v|| =', np.linalg.norm(v))


## 代码实验：遍历几个向量，对比 L1 / L2 范数

L2 范数对分量中的极端值更敏感——一个很大的分量会显著拉高 L2、但对 L1 的影响只是线性的。下面的循环展示这个差距如何随向量的「尖锐程度」变化。

In [ ]:
import numpy as np

A = np.array([[2.0, 1.0],
              [0.0, 1.0]])
vectors = [np.array([1.0, 0.0]), np.array([0.0, 1.0]), np.array([1.0, 1.0]), np.array([2.0, -1.0])]

print('A =')
print(A)
for v in vectors:
    out = A @ v
    print(f'v={v} -> A@v={out}')


## 2. 两个向量的距离 = 差向量的长度

In [ ]:
a = np.array([1.0, 2.0]); b = np.array([4.0, 6.0])
print('距离 =', np.linalg.norm(a - b))  # 5.0

## 3. ✏️ 你的任务：实现 `normalize`（缩放到长度为 1）

**推理路线**：
1. 输入 `v` 是形状 `(n,)` 的一维数组，目标是让输出向量的 L2 范数等于 1。
2. 先算 `length = np.linalg.norm(v)`——向量的「总长度」。
3. 用 `v / length` 把每个分量同比缩小：各分量比值（方向）不变，长度从 `length` 变成 `length/length = 1`。关键点：不能用 `v / np.sum(v)` 或 `v / np.max(v)`，那些不能保证 `‖result‖₂ = 1`。

**参考输入输出**：`v=[3,4]` → 范数 5，归一化后 `[0.6, 0.8]`，验证 `0.6²+0.8²=1.0`

<details><summary>点击查看参考实现</summary>

```python
return v / np.linalg.norm(v)
```

</details>

### 写 `normalize` 前明确三件事

- 输入：`v`，形状 `(n,)` 的浮点向量
- 关键步骤：计算 `length = np.linalg.norm(v)`，再做 `v / length`
- 返回：与 `v` 形状相同、L2 范数等于 1 的单位向量

In [ ]:
def normalize(v):
    # ✏️ TODO: 返回单位长度向量
    return None  # ← 改这里


In [ ]:
u = normalize(np.array([3.0, 4.0]))
print('归一化后:', u, '| 长度:', round(float(np.linalg.norm(u)), 6))
assert abs(np.linalg.norm(u) - 1.0) < 1e-9, '归一化后长度应为 1'
print('\n✅ 通过：你会归一化了。')

**🔗 Aurora 连接**：`normalize(v)` 在 Aurora 的 MFCC 流水线中对每帧特征调用，归一化后余弦相似度等价于 `np.dot(a, b)`。说话人识别比对的是 d-vector 方向而非幅度——两段录音音量差异很大，归一化后方向相近则判定为同一说话人。

## 🎨 图示：L2 范数 = 箭头长度 (3,4)→5

In [ ]:
from laviz import style, arrows2d
style()
arrows2d([[3,4]], ['v, |v|=5'], title='范数 = 长度');

In [ ]:
A = np.array([[2.0, 1.0], [0.0, 1.0]])
probes = np.array([[1,0], [0,1], [1,1], [-1,2]], dtype=float)
print('矩阵 A 会怎样移动这些向量？')
for v in probes:
    out = A @ v
    print(f'{v} -> {out} | 长度 {np.linalg.norm(v):.2f} -> {np.linalg.norm(out):.2f}')


## 参数实验：只改一个旋钮

把 `v=[3, 4]` 缩放 10 倍变成 `[30, 40]`，对比归一化结果——应仍为 `[0.6, 0.8]`，方向不随幅度改变。再把全零向量 `[0, 0]` 传入 `normalize`，观察 `nan` 或 `ZeroDivisionError`——实际代码里需要加 `eps` 保护：`v / (np.linalg.norm(v) + 1e-8)`。

In [ ]:
A = np.array([[2.0, 1.0], [0.0, 1.0]])
probes = np.array([[1,0], [0,1], [1,1], [-1,2]], dtype=float)
print('矩阵 A 会怎样移动这些向量？')
for v in probes:
    out = A @ v
    print(f'{v} -> {out} | 长度 {np.linalg.norm(v):.2f} -> {np.linalg.norm(out):.2f}')


## 本课收束

现在可以用 `compute_norm(v, p=2)` 算出 L2 长度，用 `normalize(v)` 把任意非零向量压到单位球上。Aurora 的 MFCC 流水线在每帧特征送入分类器前调用 `normalize`，让不同录音环境下的能量值落在同一尺度。MSE 损失函数 `mean((y_pred - y_true)**2)` 等价于对误差向量取 L2 范数再平方再平均。下一节的矩阵乘法中，矩阵每列的 L2 范数决定它把输入向量拉伸多少倍。

In [ ]:
# 小检查：先看 shape，再做运算
import numpy as np

a = np.array([1.0, 2.0])
b = np.array([3.0, 4.0])
A = np.array([[1.0, 2.0], [0.0, 1.0]])

print('a.shape =', a.shape)
print('b.shape =', b.shape)
print('A.shape =', A.shape)
print('a + b =', a + b)
print('A @ a =', A @ a)
print('a dot b =', float(a @ b))
